In [1]:
#Inicio de clase

from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ED_2026_02_v2
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/drive/MyDrive/ED_2026_02_v2')

Mounted at /content/drive
/content/drive/MyDrive/ED_2026_02_v2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 86.9 MB/s eta 0:00:00


# CH05 — Array-Based Sequences

Material del curso basado en Goodrich, Tamassia & Goldwasser. El código de ejemplo está en `goodrich/ch05`.

## Resumen del capítulo 5 — Array-Based Sequences (Goodrich, Tamassia & Goldwasser)

### 5.1 Los tipos secuencia de Python: `list`, `tuple`, `str`

Python ofrece tres tipos secuencia integrados basados en arreglos: `list` (mutable, tamaño variable), `tuple` (inmutable, tamaño fijo) y `str` (inmutable, secuencia de caracteres). Los tres se apoyan internamente en un **arreglo de bajo nivel**: un bloque de memoria consecutivo donde cada celda ocupa el mismo número de bytes, lo que permite acceso en **O(1)** a cualquier índice mediante aritmética de direcciones (`dirección = inicio + k * tamaño_celda`).

Una diferencia clave frente a un arreglo de C: en Python cada celda del arreglo no guarda el objeto en sí, sino una **referencia** (puntero) al objeto real, que puede vivir en cualquier parte de la memoria y tener un tamaño distinto. Esto es lo que permite que una misma lista contenga objetos de tipos y tamaños distintos.

### 5.2 Arreglos referenciados y arreglos compactos

- **Arreglo referenciado** (*referential array*): cada celda apunta a un objeto externo. Es lo que usa `list` en Python. Ventaja: cada elemento puede tener tamaño distinto. Consecuencia: copiar una lista con `list(otra)` o mediante *slicing* copia las **referencias**, no los objetos (copia superficial / *shallow copy*).
- **Arreglo compacto**: los propios datos (bytes crudos) se almacenan consecutivamente dentro del arreglo, sin indirección. Ejemplo: el módulo `array` de Python, o cadenas de texto (`str`), que en CPython se almacenan como secuencias compactas de caracteres, no como listas de referencias a objetos-carácter individuales. Ventajas frente al arreglo referenciado: mejor localidad de memoria (más eficiente en caché) y no hay que dereferenciar cada elemento.

### 5.3 Arreglos dinámicos (*dynamic arrays*) y amortización

Una lista de Python puede crecer (`append`) aunque un arreglo de bajo nivel tenga tamaño fijo al crearse. La estrategia es la de **arreglo dinámico**: se reserva un arreglo de **capacidad** mayor que el número de elementos usados (`n`); cuando se llena, se crea un arreglo nuevo más grande, se copian los elementos existentes y se descarta el arreglo viejo.

`dynamic_array.py` implementa esta idea en la clase `DynamicArray`:

```python
class DynamicArray:
    def __init__(self):
        self._n = 0
        self._capacity = 1
        self._A = self._make_array(self._capacity)

    def append(self, obj):
        if self._n == self._capacity:      # sin espacio
            self._resize(2 * self._capacity)  # duplicar capacidad
        self._A[self._n] = obj
        self._n += 1
```

**¿Por qué duplicar (en vez de sumar una cantidad fija)?** Si cada vez que se llena el arreglo se aumenta la capacidad en una constante fija, el costo total de *n* inserciones es **O(n²)** (se reserva/copia con demasiada frecuencia). Si en cambio se **duplica** la capacidad cada vez, el costo total de *n* inserciones es **O(n)**: aunque una operación individual de `append` puede costar O(n) cuando toca redimensionar, esos redimensionamientos son cada vez más espaciados (después de una copia de tamaño *c*, hay que hacer *c* inserciones antes de la siguiente copia). Esto se conoce como **costo amortizado O(1) por operación** (análisis por el *método contable*: se "cobra" un poco más por cada `append` barato para pagar por adelantado el siguiente redimensionamiento).

`experiment_list_append.py` y `experiment_list_size.py` verifican esto experimentalmente: el tiempo *promedio* por `append` se mantiene aproximadamente constante a medida que crece `n`, y `sys.getsizeof(lista)` crece a saltos (no de a uno) a medida que se insertan elementos, evidenciando la estrategia de sobre-reserva de capacidad.

In [ ]:
import ctypes                                      # provides low-level arrays

class DynamicArray:
  """A dynamic array class akin to a simplified Python list."""

  def __init__(self):
    """Create an empty array."""
    self._n = 0                                    # count actual elements
    self._capacity = 1                             # default array capacity
    self._A = self._make_array(self._capacity)     # low-level array

  def __len__(self):
    """Return number of elements stored in the array."""
    return self._n

  def __getitem__(self, k):
    """Return element at index k."""
    if not 0 <= k < self._n:
      raise IndexError('invalid index')
    return self._A[k]                              # retrieve from array

  def append(self, obj):
    """Add object to end of the array."""
    if self._n == self._capacity:                  # not enough room
      self._resize(2 * self._capacity)             # so double capacity
    self._A[self._n] = obj
    self._n += 1

  def _resize(self, c):                            # nonpublic utitity
    """Resize internal array to capacity c."""
    B = self._make_array(c)                        # new (bigger) array
    for k in range(self._n):                       # for each existing value
      B[k] = self._A[k]
    self._A = B                                    # use the bigger array
    self._capacity = c

  def _make_array(self, c):                        # nonpublic utitity
     """Return new array with capacity c."""
     return (c * ctypes.py_object)()               # see ctypes documentation

  def insert(self, k, value):
    """Insert value at index k, shifting subsequent values rightward."""
    # (for simplicity, we assume 0 <= k <= n in this verion)
    if self._n == self._capacity:                  # not enough room
      self._resize(2 * self._capacity)             # so double capacity
    for j in range(self._n, k, -1):                # shift rightmost first
      self._A[j] = self._A[j-1]
    self._A[k] = value                             # store newest element
    self._n += 1

  def remove(self, value):
    """Remove first occurrence of value (or raise ValueError)."""
    # note: we do not consider shrinking the dynamic array in this version
    for k in range(self._n):
      if self._A[k] == value:              # found a match!
        for j in range(k, self._n - 1):    # shift others to fill gap
          self._A[j] = self._A[j+1]
        self._A[self._n - 1] = None        # help garbage collection
        self._n -= 1                       # we have one less item
        return                             # exit immediately
    raise ValueError('value not found')    # only reached if no match

In [ ]:
A = ["A", 1,"R", "L"]


In [ ]:
A[0]

'A'

DynamicArray

In [ ]:
class nuevo(DynamicArray):
  def __getitem__(self, k):
    if not - self._n <= k < self._n:
      raise IndexError('invalid index')
    if k < 0:
      k += self._n
    return self._A[k]


In [ ]:
L2 = nuevo()

Agregar elementos al DynamicArray

In [ ]:
for i in "A1RL":
  L2.append(i)


In [ ]:
#Extraer el elemento de la penultima posición
L2[-2]

'R'

In [ ]:
L = DynamicArray()

In [ ]:
for i in "ARL":
  L.append(i)


In [ ]:
# Agregar un elemento en el DynamicArray en una posición n
L.insert(1, 1)
for i in range(len(L)):
  print(L[i], end = "")

A1RL

In [ ]:
# Función de longitud - capacidad
len(L), L._capacity

(4, 4)

### 5.4 Eficiencia de los tipos secuencia de Python

| Operación | `list` | Notas |
|---|---|---|
| `len(data)` | O(1) | se guarda el conteo, no se recorre |
| `data[j]` (leer/escribir) | O(1) | acceso directo por índice |
| `data.append(value)` | O(1) *amortizado* | ver §5.3 |
| `data.pop()` (quitar el último) | O(1) *amortizado* | |
| `data.insert(k, value)` | O(n) | hay que desplazar los elementos desde `k` en adelante |
| `data.pop(k)` / `del data[k]` | O(n) | hay que desplazar los elementos posteriores a `k` |
| `data.remove(value)` | O(n) | primero busca (O(n)), luego desplaza (O(n)) |
| `value in data` | O(n) | búsqueda lineal |
| `data1 == data2` (comparación elemento a elemento) | O(n) | |
| *slicing* `data[a:b]` | O(b-a) | crea una lista nueva |
| `data1 + data2` | O(n1+n2) | crea una lista nueva |
| `data * k` | O(nk) | |

Idea clave: las operaciones que agregan/quitan **al final** del arreglo son O(1) amortizado; las que agregan/quitan/insertan en una posición arbitraria (especialmente al inicio) son O(n), porque obligan a desplazar elementos para mantenerlos contiguos en memoria.

### 5.5 Ejemplos de uso de secuencias basadas en arreglos

- **`caesar.py`** — cifrado César: usa un arreglo compacto de 26 letras como tabla de sustitución cíclica (`chr`, `ord`, aritmética módulo 26) para cifrar/descifrar un mensaje desplazando cada letra un número fijo de posiciones.
- **`insertion_sort.py`** — ordenamiento por inserción **in-place** sobre una lista: mantiene un prefijo ya ordenado y va insertando cada nuevo elemento en su posición correcta dentro de ese prefijo, desplazando los mayores hacia la derecha. Peor caso O(n²), pero no requiere memoria adicional.
- **`high_scores.py`** (`GameEntry` / `Scoreboard`) — mantiene un arreglo de **tamaño fijo** con los mejores puntajes en orden no creciente; al agregar un nuevo puntaje, se desplazan los puntajes menores para abrir espacio (misma técnica de desplazamiento que `insertion_sort`). Ilustra el uso de un arreglo como estructura de tamaño fijo/capacidad acotada.
- **`tic_tac_toe.py`** — tablero como **lista bidimensional** (`list` de `list`s, 3×3): ejemplo de arreglo multidimensional construido a partir de listas de Python, con cuidado de no compartir referencias entre filas (`[[0]*3 for _ in range(3)]` en vez de `[[0]*3]*3`).

### 5.6 Ideas clave para recordar

1. Un arreglo de bajo nivel da acceso O(1) a cualquier índice porque las celdas son consecutivas y del mismo tamaño; en Python cada celda guarda una **referencia**, no el objeto.
2. `list`, `tuple` y `str` son arreglos; `list` es *referencial* y mutable, `tuple`/`str` son inmutables (y `str` es *compacta*).
3. Un arreglo dinámico crece **duplicando** su capacidad cuando se llena, logrando `append` en O(1) **amortizado**; crecer de a una cantidad fija daría O(n²) total.
4. Operar en el **extremo derecho** de una lista (`append`, `pop`) es barato (O(1) amortizado); operar al **inicio o en el medio** (`insert(0, ...)`, `pop(0)`, `remove`) es caro (O(n)) porque hay que desplazar elementos para mantener la contigüidad.
5. Copiar una lista (por *slicing* o `list(...)`) es una copia **superficial**: copia las referencias, no los objetos apuntados.

## Conceptos teóricos: verificación con código

A continuación se importan y ejecutan los ejemplos de `goodrich/ch05` para verificar el comportamiento descrito arriba.

In [ ]:
import sys
sys.path.append(r'C:\Users\fator\Desktop\Proyectos1\Estructura_Datos\2026_02')
from goodrich.ch05.dynamic_array import DynamicArray
from goodrich.ch05.high_scores import GameEntry, Scoreboard
from goodrich.ch05.insertion_sort import insertion_sort

In [ ]:
# DynamicArray se comporta como una lista simplificada pero es de tipo object
da = DynamicArray()
for x in [10, 20, 30, 40, 50]:
    da.append(x)

len(da), [da[i] for i in range(len(da))]

(5, [10, 20, 30, 40, 50])

### Costo amortizado: capacidad vs. número de elementos

`sys.getsizeof` sobre una lista real de Python muestra cómo la memoria reservada crece a saltos (no de a uno), evidenciando la estrategia de sobre-reserva por duplicación.

In [ ]:
import sys
data = []
for i in range(15):
    size = sys.getsizeof(data)
    print(f'n = {i:3d} bytes ={size}')
    data.append(None)

n =   0 bytes =56
n =   1 bytes =88
n =   2 bytes =88
n =   3 bytes =88
n =   4 bytes =88
n =   5 bytes =120
n =   6 bytes =120
n =   7 bytes =120
n =   8 bytes =120
n =   9 bytes =184
n =  10 bytes =184
n =  11 bytes =184
n =  12 bytes =184
n =  13 bytes =184
n =  14 bytes =184


In [ ]:
data = []
prev_size = -1
for k in range(15):
    size = sys.getsizeof(data)
    if size != prev_size:
        print(f"n = {k:2d}  bytes = {size}")
        prev_size = size
    data.append(None)

n =  0  bytes = 56
n =  1  bytes = 88
n =  5  bytes = 120
n =  9  bytes = 184


In [ ]:
# Scoreboard: se mantiene ordenado de mayor a menor al ir agregando puntajes
board = Scoreboard(5)
for nombre, puntaje in [('Rob', 750), ('Mike', 1105), ('Rose', 590), ('Jill', 740), ('Jack', 510), ('Anna', 660)]:
    board.add(GameEntry(nombre, puntaje))

print(board)

(Mike, 1105)
(Rob, 750)
(Jill, 740)
(Anna, 660)
(Rose, 590)


In [ ]:
# insertion_sort ordena in-place
lista = [5, 2, 9, 1, 7, 3]
insertion_sort(lista)
lista

[1, 2, 3, 5, 7, 9]

# Ejercicios en clase

## Ejercicio 1 — Insertar en una lista ordenada

Dada una lista ya ordenada de forma ascendente y un nuevo elemento, insértelo en la **posición correcta** para que la lista resultante quede ordenada.

**Restricciones:** no puede usar `sort`, `sorted`, `list.insert`, ni `bisect`. Solo puede recorrer la lista, comparar elementos y construir el resultado con las operaciones básicas de lista (indexación, `append`, slicing manual si lo necesita).

Programe `insertar_ordenado(L, x)`, que retorna una **nueva lista** con `x` insertado en la posición correcta (no es necesario modificar `L` in-place).

In [ ]:
def insertar_ordenado(L, x):
    resultado = []
    insertado = False
    # Si x es menor al elemento
    for elemento in L:
      if not insertado and x <= elemento:
        resultado.append(x)
        insertado = True
     # Si x es mayor a todos los elementos
      resultado.append(elemento)
    if not insertado:
      resultado.append(x)
    return resultado


In [ ]:
casos = [
    ([1, 3, 5, 7], 4),
    ([1, 3, 5, 7], 0),
    ([1, 3, 5, 7], 8),
    ([], 5),
    ([2, 2, 4], 2),
]
for L, x in casos:
    resultado = insertar_ordenado(L, x)
    esperado = sorted(L + [x])
    print(L, '+', x, '->', resultado, 'esperado:', esperado)

[1, 3, 5, 7] + 4 -> [1, 3, 4, 5, 7] esperado: [1, 3, 4, 5, 7]
[1, 3, 5, 7] + 0 -> [0, 1, 3, 5, 7] esperado: [0, 1, 3, 5, 7]
[1, 3, 5, 7] + 8 -> [1, 3, 5, 7, 8] esperado: [1, 3, 5, 7, 8]
[] + 5 -> [5] esperado: [5]
[2, 2, 4] + 2 -> [2, 2, 2, 4] esperado: [2, 2, 2, 4]


**Pregunta:** ¿cuál es la complejidad computacional (tiempo) de `insertar_ordenado` en el peor caso, en función de $n = len(L)$? Justifique en términos de cuántos elementos hay que recorrer/desplazar.

Respuesta:
Complejidad: O(n)
Justificacion: Tanto en el peor como en el mejor caso es necesario recorrer los n elementos de la lista, de acuerdo con esto el tiempo aumenta proporcionalmente al tamaño de la lista.


## Ejercicio 2 — Complejidad de `insertar_ordenado`

Verifique experimentalmente la complejidad del Ejercicio 1: mida el tiempo de `insertar_ordenado` sobre listas ordenadas de tamaño creciente (por ejemplo $n = 1000, 2000, 4000, 8000$), insertando siempre en el **peor caso** (por ejemplo, un elemento mayor que todos, o menor que todos, según en qué extremo su implementación recorre primero).

Grafique o imprima el tiempo por tamaño y comente si el crecimiento es consistente con la respuesta del Ejercicio 1.

In [ ]:
from time import time

for n in [1000, 2000, 4000, 8000]:
    L = list(range(0, 2*n, 2))  # ordenada, pares
    x = -1  # peor caso: menor que todos
    a = time()
    insertar_ordenado(L, x)
    b = time()
    print(f'n = {n:6d}  tiempo = {b-a:.6f} s')

n =   1000  tiempo = 0.000051 s
n =   2000  tiempo = 0.000109 s
n =   4000  tiempo = 0.000140 s
n =   8000  tiempo = 0.000299 s


## Ejercicio 3

Usar herencia para modificar `DynamicArray` y proponer un método `resize1` que aumente el espacio de memoria en tres unidades. Luego comparar cual versión es más rápida.

In [ ]:
# Solución 1
class DynamicArray1(DynamicArray):
    """Dynamic array que aumenta su capacidad en 3 unidades."""

    def _resize(self, c):
        """Redimensiona el arreglo interno a capacidad c."""
        c = self._capacity + 3
        B = self._make_array(c)

        for k in range(self._n):
            B[k] = self._A[k]

        self._A = B
        self._capacity = c

In [ ]:
# Solución 2
class DynamicArray2(DynamicArray):
  def append(self, obj):
    """Add object to end of the array."""
    if self._n == self._capacity:                  # not enough room
      self._resize(2 * self._capacity)             # so double capacity
    self._A[self._n] = obj
    self._n += 1

  def insert(self, k, value):
    """Insert value at index k, shifting subsequent values rightward."""
    # (for simplicity, we assume 0 <= k <= n in this verion)
    if self._n == self._capacity:                  # not enough room
      self._resize(2 * self._capacity)             # so double capacity
    for j in range(self._n, k, -1):                # shift rightmost first
      self._A[j] = self._A[j-1]
    self._A[k] = value                             # store newest element
    self._n += 1


Tiempo de ejecución

In [ ]:
from time import time

L1 = DynamicArray1()
a = time()
for i in range(10000):
  L1.append(i)
b = time()
print("Tiempo de ejecución de L1:", b-a)


Tiempo de ejecución de L1: 11.511861801147461


In [ ]:
L2 = DynamicArray2()
a = time()
for i in range(10000):
  L2.append(i)
b = time()
print("Tiempo de ejecución de L2:", b-a)

Tiempo de ejecución de L2: 0.024544477462768555


## Ejercicio 4: Goodrich

- 5.4, 5.6, 5.16

In [ ]:
class nuevo(DynamicArray):
  def __getitem__(self, k):
    if not - self._n <= k < self._n:
      raise IndexError('invalid index')
    if k < 0:
      k += self._n
    return self._A[k]


In [ ]:
L2 = nuevo()
for i in "A1RL":
  L2.append(i)

In [ ]:
L2[-2]

'R'

5.6

In [ ]:
class insertar(DynamicArray):
  def insert(self, k, value):
    if self._n == self._capacity:
      self._resize(self._capacity + 3)
    for j in range(self._n, k, -1):
      self._A[j] = self._A[j-1]
    self._A[k] = value
    self._n += 1

In [ ]:
class Insertar2(DynamicArray):
    def insert(self, k, value):
        if k < 0 or k > self._n:
            raise IndexError("Índice fuera de rango")

        if self._n == self._capacity:
            self._resize(max(1, 2 * self._capacity))

        for j in range(self._n, k, -1):
            self._A[j] = self._A[j - 1]

        self._A[k] = value
        self._n += 1

In [ ]:
a = insertar()
b = Insertar2()

for i in range(100):
    a.insert(i, i)
    b.insert(i, i)

print("Original:")
print("n =", a._n)
print("capacity =", a._capacity)

print("\nOptimizada:")
print("n =", b._n)
print("capacity =", b._capacity)

Original:
n = 100
capacity = 100

Optimizada:
n = 100
capacity = 128


5.16

In [ ]:
class dynamic_array3(DynamicArray):
  def pop(self):
        """Remove and return the last element of the array."""
        if self._n == 0:
            raise IndexError('pop from empty array')

        value = self._A[self._n - 1]
        self._A[self._n - 1] = None
        self._n -= 1

        if self._n < self._capacity // 4:
            self._resize(self._capacity // 2)

        return value

In [ ]:
a = dynamic_array3()

# Agregamos elementos
a.append(10)
a.append(20)
a.append(30)
a.append(40)

print("Elementos:", len(a))
print("Capacidad:", a._capacity)

# Eliminamos el último elemento
elemento = a.pop()

print("Elemento eliminado:", elemento)
print("Elementos:", len(a))
print("Capacidad:", a._capacity)

Elementos: 4
Capacidad: 4
Elemento eliminado: 40
Elementos: 3
Capacidad: 4


In [2]:
#Final de clase "guardar cambios"
from google.colab import userdata

usuario_github = "Johane-salazarr"
repo = "ED_2026_02_v2"
token = userdata.get('GITHUB_TOKEN').strip()

%cd /content/drive/MyDrive/{repo}
!git config --global user.email "johan.salazar1@est.uexternado.edu.co"
!git config --global user.name "Johan Salazar"
!git add student_work/
!git commit -m "ch05_teoria"
!git push https://{token}@github.com/{usuario_github}/{repo}.git main

/content/drive/MyDrive/ED_2026_02_v2
[main c7f9c32] ch05_teoria
 2 files changed, 2 insertions(+), 2 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.01 KiB | 206.00 KiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Johane-salazarr/ED_2026_02_v2.git
   7a25cc0..c7f9c32  main -> main
